# Evaluate CelebDF-v1 Level 4 RAL/Linear/Ensemble Per-Sample Probabilities

Notebook này chạy riêng **CelebDF-v1 corruption level 4** cho các corruption trong `CORRUPTIONS`.

Output gồm 2 file:
- Per-sample CSV: mỗi dòng là một mẫu, có `ral_prob`, `linear_prob`, và ensemble probabilities `alpha * ral_prob + (1-alpha) * linear_prob` với alpha `0.2, 0.4, 0.6, 0.8`.
- Summary CSV: metric `acc`, `f1`, `auc`, `ap`, `eer` cho từng dataset và từng cột xác suất/method.

Cần attach thêm dataset source real local-token shards và linear probe model.


## Kaggle Setup

In [ ]:
# !git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
# %cd /kaggle/working/training-free-tta-for-deepfake-detection
# !pip install -q -e . --no-deps
# !pip install -q open_clip_torch
!find /kaggle/input -maxdepth 3 -type d | sort | sed -n '1,220p'

## Imports

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.mixture import GaussianMixture
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

repo_root = Path.cwd()
sys.path.insert(0, str(repo_root / 'code' if (repo_root / 'code').exists() else repo_root))
from deepfake_tta.modeling import LinearProbe, OSDLinearProbe


## Config

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42

# Source real local-token shards. Đổi path theo Kaggle dataset m attach.
SOURCE_TOKEN_DIR = Path('/kaggle/input/ral-clip-local-tokens/ral_clip_local_tokens')
SOURCE_SHARD_GLOB = '*_shard*.pt'
LOCAL_LAYERS_OVERRIDE = [11]  # CLIP ViT-L/14 block 12, zero-based index. Set None to use shard metadata.

# Linear probe model. Có thể dùng OSD model nếu muốn, loader tự nhận dạng.
LINEAR_MODEL_PATH = Path('/kaggle/input/ffpp-training-free-models/ffpp_linear_probe_split.pt')
LINEAR_PRED_THRESHOLD = 0.5
ENSEMBLE_ALPHAS = [0.2, 0.4, 0.6, 0.8]

RETRIEVE_M = 16
TOP_K_PATCHES = 16
NEIGHBOR_RADIUS = 1
BATCH_SIZE = 16
NUM_WORKERS = 2
USE_AMP = True
MAX_TARGET = None
SHUFFLE_TARGET = True
RUN_TARGET_EXPANSION = True
EXPAND_SCORE_QUANTILE = 0.20
EXPAND_MAX_AUG_VAR = 1e-4
MAX_TARGET_REAL_ADD = 512
RAL_THRESHOLD_MODE = 'quantile_prior'  # quantile_prior | gmm_non_safe | gmm_all
RAL_FAKE_PRIOR = 0.87

CORRUPTIONS = ['color_contrast', 'color_saturation', 'gaussian_blur', 'resize']
CELEB_LEVEL = 4

OUTPUT_DIR = Path('/kaggle/working/celebdfv1_level4_ral_linear_ensemble')
SAMPLES_OUTPUT = OUTPUT_DIR / 'celebdfv1_level4_ral_linear_ensemble_samples.csv'
SUMMARY_OUTPUT = OUTPUT_DIR / 'celebdfv1_level4_ral_linear_ensemble_summary.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print('device:', DEVICE)

## Path Resolution

In [ ]:
def first_existing(candidates):
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path
    return Path(candidates[0])

def first_existing_or_none(candidates):
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path
    return None

CSV_PATH = first_existing([
    '/kaggle/input/datasets/jamestashvik/deepfakebench/deepfakebench_dataset.csv',
    '/kaggle/input/deepfakebench/deepfakebench_dataset.csv',
])
DEEPFAKEBENCH_ROOT = first_existing([
    '/kaggle/input/datasets/jamestashvik/deepfakebench/DeepFakeBench',
    '/kaggle/input/deepfakebench/DeepFakeBench',
])

# Official FaceForensics++ split jsons. Theo DeepFakeBench Kaggle của m,
# train.json/val.json/test.json nằm ngay trong DeepFakeBench/FaceForensics++.
SPLIT_ROOT = DEEPFAKEBENCH_ROOT / 'FaceForensics++'

def has_dataset_anchors(path, dataset_name):
    path = Path(path)
    if dataset_name == 'Celeb-DF-v1':
        anchors = ('Celeb-real', 'YouTube-real', 'Celeb-synthesis')
    else:
        anchors = ('original_sequences', 'manipulated_sequences')
    return any((path / anchor).exists() for anchor in anchors)

def find_dataset_root(root, dataset_name):
    root = Path(root)
    for candidate in [root / dataset_name, root]:
        if has_dataset_anchors(candidate, dataset_name):
            return candidate
    return root

def resolve_corruption_root(base_roots, dataset_name, corruption, level):
    candidates = []
    for root in base_roots:
        root = Path(root)
        candidates.extend([
            root / corruption / f'level_{level}' / dataset_name,
            root / corruption / f'level_{level}',
            root / corruption / dataset_name,
            root / corruption,
            root / f'level_{level}' / dataset_name,
            root / f'level_{level}',
            root / dataset_name,
            root,
        ])
    for candidate in candidates:
        if has_dataset_anchors(candidate, dataset_name):
            return candidate
    return candidates[0]

def celeb_level_roots(level):
    return [
        f'/kaggle/input/datasets/elisevo/celebdfv1-level-{level}/processed_output',
        f'/kaggle/input/datasets/vohoanghoavien/celebdfv1-level-{level}/processed_output',
        f'/kaggle/input/celebdfv1-level-{level}/processed_output',
    ]

def ffpp_level_roots(level, corruption):
    if level == 5:
        pretty = corruption.replace('_', '-')
        return [
            f'/kaggle/input/ff-{pretty}-5',
            f'/kaggle/input/datasets/vohoanghoavien/ff-{pretty}-5',
        ]
    return [
        f'/kaggle/input/ffpp-corruption-level-{level}',
        f'/kaggle/input/ff-corruption-level-{level}',
        f'/kaggle/input/ff-corruption-level{level}',
        f'/kaggle/input/datasets/elisevo/ffpp-corruption-level-{level}',
        f'/kaggle/input/datasets/elisevo/ff-corruption-level{level}',
        f'/kaggle/input/datasets/vhonghoavin/ff-corruption-level-{level}',
    ]

print('CSV_PATH:', CSV_PATH)
print('DEEPFAKEBENCH_ROOT:', DEEPFAKEBENCH_ROOT)
print('SPLIT_ROOT:', SPLIT_ROOT)


## Target Specs

In [ ]:
TARGET_SPECS = []

for corruption in CORRUPTIONS:
    root = resolve_corruption_root(celeb_level_roots(CELEB_LEVEL), 'Celeb-DF-v1', corruption, CELEB_LEVEL)
    TARGET_SPECS.append({
        'name': f'celebdfv1-level{CELEB_LEVEL}-{corruption}',
        'dataset_name': 'Celeb-DF-v1',
        'split': None,
        'replacement_root': root,
        'level': CELEB_LEVEL,
        'corruption': corruption,
    })

pd.DataFrame(TARGET_SPECS)[['name', 'dataset_name', 'split', 'replacement_root']]


## Load Source And Model

In [ ]:
def load_source_memory(source_dir, pattern):
    paths = sorted(Path(source_dir).glob(pattern))
    if not paths:
        raise FileNotFoundError(f'No source token shards found: {source_dir}/{pattern}')
    globals_, labels_, all_paths = [], [], []
    locals_by_key = None
    metadata = None
    for path in paths:
        payload = torch.load(path, map_location='cpu')
        if metadata is None:
            metadata = payload.get('metadata', {})
            locals_by_key = {key: [] for key in payload['local']}
        globals_.append(payload['global'].float())
        labels_.append(payload['labels'].long())
        all_paths.extend(payload.get('paths', []))
        for key, value in payload['local'].items():
            locals_by_key[key].append(value.float())
    labels = torch.cat(labels_).long()
    real_idx = torch.where(labels.eq(0))[0]
    memory = {
        'global': F.normalize(torch.cat(globals_).float(), dim=-1)[real_idx].contiguous(),
        'local': {key: F.normalize(torch.cat(parts).float(), dim=-1)[real_idx].contiguous() for key, parts in locals_by_key.items()},
        'labels': labels[real_idx],
        'metadata': metadata or {},
    }
    print('source shards:', len(paths), '| label counts:', torch.bincount(labels, minlength=2).tolist())
    print('source real memory:', tuple(memory['global'].shape))
    for key, value in memory['local'].items():
        print(key, tuple(value.shape))
    return memory

def load_probe_model(model_path, dim, device):
    state = torch.load(model_path, map_location='cpu')
    if 'center' in state and 'basis' in state:
        model = OSDLinearProbe(dim, center=state['center'], basis=state['basis'])
        model_type = 'osd_linear_probe'
    else:
        model = LinearProbe(dim)
        model_type = 'linear_probe'
    model.load_state_dict(state)
    model.to(device).eval()
    print('loaded model:', model_type, model_path)
    return model_type, model

source_memory = load_source_memory(SOURCE_TOKEN_DIR, SOURCE_SHARD_GLOB)
SOURCE_METADATA = source_memory['metadata']
clip_id = SOURCE_METADATA.get('clip_model', 'ViT-L-14/openai')
CLIP_MODEL, PRETRAINED = clip_id.split('/')[0], clip_id.split('/')[1]
SOURCE_LAYERS = SOURCE_METADATA.get('layers', [-6])
LAYERS = SOURCE_LAYERS if LOCAL_LAYERS_OVERRIDE is None else LOCAL_LAYERS_OVERRIDE
missing_layer_keys = [f'layer_{layer}' for layer in LAYERS if f'layer_{layer}' not in source_memory['local']]
if missing_layer_keys:
    raise ValueError(
        f'Source token shards do not contain requested local layers {missing_layer_keys}. '
        f'Available keys: {sorted(source_memory["local"].keys())}. '
        'Regenerate/attach ral_clip_local_tokens with LOCAL_LAYERS_OVERRIDE, or set LOCAL_LAYERS_OVERRIDE = None.'
    )
PATCH_KEEP_MODE = SOURCE_METADATA.get('patch_keep_mode', 'all')
PATCH_STRIDE = int(SOURCE_METADATA.get('patch_stride', 1) or 1)
PATCH_CENTER_FRACTION = float(SOURCE_METADATA.get('patch_center_fraction', 1.0) or 1.0)
linear_model_type, linear_model = load_probe_model(LINEAR_MODEL_PATH, source_memory['global'].shape[1], DEVICE)
print('target extraction:', CLIP_MODEL, PRETRAINED, LAYERS, PATCH_KEEP_MODE, PATCH_STRIDE)

## Data And Extractor Helpers

In [ ]:
def prepare_generic_dataframe(csv_path, dataset_name, deepfakebench_root, replacement_root=None):
    df = pd.read_csv(csv_path)
    df = df[df['datasetname'].eq(dataset_name)].copy()
    df['label_num'] = df['label'].map({'REAL': 0, 'FAKE': 1}).astype(int)
    df['imagepath_fixed'] = df['imagepath'].astype(str).str.replace('../input/deepfakebench', str(deepfakebench_root), regex=False)
    if replacement_root is not None:
        source_root = Path(deepfakebench_root) / dataset_name
        target_root = find_dataset_root(replacement_root, dataset_name)
        df['imagepath_fixed'] = df['imagepath_fixed'].astype(str).str.replace(str(source_root), str(target_root), regex=False)
    exists = df['imagepath_fixed'].map(lambda p: Path(p).exists())
    missing = int((~exists).sum())
    if missing:
        print(f'dropping missing files: {missing}/{len(df)} | {dataset_name}')
        df = df[exists].copy()
    return df.reset_index(drop=True)


def extract_ffpp_video_key(path):
    parts = Path(path).parts
    if 'frames' in parts:
        idx = parts.index('frames')
        if idx + 1 < len(parts):
            return parts[idx + 1]
    return Path(path).parent.name

def is_original_ffpp_path(path):
    return 'original_sequences' in path

def prepare_ffpp_split_dataframe_exact(csv_path, deepfakebench_root, split_name, split_root):
    split_path = Path(split_root) / f'{split_name}.json'
    if not split_path.exists():
        raise FileNotFoundError(
            f'Missing FF++ split file: {split_path}\n'
            'Set SPLIT_ROOT to the one folder containing official FaceForensics++ test.json.'
        )
    print('using FF++ split json:', split_path)
    with split_path.open('r') as f:
        pairs = json.load(f)
    video_ids = {str(vid) for pair in pairs for vid in pair}
    pair_keys = {f'{a}_{b}' for a, b in pairs} | {f'{b}_{a}' for a, b in pairs}

    df = pd.read_csv(csv_path)
    df = df[df['datasetname'].eq('FaceForensics++')].copy()
    df['label_num'] = df['label'].map({'REAL': 0, 'FAKE': 1}).astype(int)
    df['imagepath_fixed'] = df['imagepath'].astype(str).str.replace('../input/deepfakebench', str(deepfakebench_root), regex=False)
    df['video_key'] = df['imagepath_fixed'].map(extract_ffpp_video_key)
    df['is_original'] = df['imagepath_fixed'].map(is_original_ffpp_path)

    original_mask = df['is_original'] & df['video_key'].isin(video_ids)
    fake_pair_mask = (~df['is_original']) & df['video_key'].isin(pair_keys)
    fake_component_mask = (~df['is_original']) & df['video_key'].str.split('_').map(
        lambda x: len(x) >= 2 and x[0] in video_ids and x[1] in video_ids
    )
    out = df[original_mask | fake_pair_mask | fake_component_mask].reset_index(drop=True)
    print('FF++ split dataframe:', split_name, out.shape, '| counts:', out['label_num'].value_counts().sort_index().to_dict())
    return out

def build_target_df(spec):
    if spec['dataset_name'] == 'FaceForensics++':
        df = prepare_ffpp_split_dataframe_exact(CSV_PATH, DEEPFAKEBENCH_ROOT, spec.get('split') or 'test', split_root=SPLIT_ROOT)
        if spec.get('replacement_root') is not None:
            source_root = Path(DEEPFAKEBENCH_ROOT) / 'FaceForensics++'
            target_root = find_dataset_root(spec['replacement_root'], 'FaceForensics++')
            df['imagepath_fixed'] = df['imagepath_fixed'].astype(str).str.replace(str(source_root), str(target_root), regex=False)
    else:
        df = prepare_generic_dataframe(CSV_PATH, spec['dataset_name'], DEEPFAKEBENCH_ROOT, spec.get('replacement_root'))
    if MAX_TARGET is not None and len(df) > MAX_TARGET:
        df = df.sample(n=MAX_TARGET, random_state=SEED).reset_index(drop=True)
    if SHUFFLE_TARGET:
        df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    return df

class TargetImageDataset(Dataset):
    def __init__(self, dataframe, preprocess, augment=None):
        self.df = dataframe.reset_index(drop=True)
        self.preprocess = preprocess
        self.augment = augment
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['imagepath_fixed']).convert('RGB')
        if self.augment == 'hflip':
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        return self.preprocess(image), int(row['label_num']), row['imagepath_fixed']

def patch_keep_indices(num_patches):
    side = int(round(math.sqrt(num_patches)))
    if side * side != num_patches or PATCH_KEEP_MODE == 'all':
        return torch.arange(num_patches, dtype=torch.long)
    if PATCH_KEEP_MODE == 'stride':
        return torch.tensor([r * side + c for r in range(0, side, PATCH_STRIDE) for c in range(0, side, PATCH_STRIDE)], dtype=torch.long)
    if PATCH_KEEP_MODE == 'center':
        keep_side = max(1, int(round(side * PATCH_CENTER_FRACTION)))
        start = max(0, (side - keep_side) // 2)
        end = min(side, start + keep_side)
        return torch.tensor([r * side + c for r in range(start, end) for c in range(start, end)], dtype=torch.long)
    raise ValueError(PATCH_KEEP_MODE)

class OpenClipLocalExtractor:
    def __init__(self, model, layers):
        self.model = model.eval(); self.layers = list(layers); self.captures = {}; self.handles = []
        blocks = self.model.visual.transformer.resblocks
        n_blocks = len(blocks)
        self.layer_indices = [layer if layer >= 0 else n_blocks + layer for layer in self.layers]
        for idx in self.layer_indices:
            self.handles.append(blocks[idx].register_forward_hook(self._hook(idx)))
    def _hook(self, idx):
        def hook(module, inputs, output):
            self.captures[idx] = (output[0] if isinstance(output, tuple) else output).detach()
        return hook
    def close(self):
        for h in self.handles: h.remove()
    @torch.inference_mode()
    def __call__(self, images):
        self.captures = {}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(USE_AMP and images.device.type == 'cuda')):
            global_features = self.model.encode_image(images)
        global_features = F.normalize(global_features.float(), dim=-1).cpu()
        locals_by_key = {}
        batch_size = images.shape[0]
        for raw_layer, idx in zip(self.layers, self.layer_indices):
            tokens = self.captures[idx].float()
            if tokens.shape[1] == batch_size:
                tokens = tokens.permute(1, 0, 2).contiguous()
            patch_tokens = F.normalize(tokens[:, 1:, :], dim=-1)
            locals_by_key[f'layer_{raw_layer}'] = patch_tokens[:, patch_keep_indices(patch_tokens.shape[1]), :].cpu().contiguous()
        return global_features, locals_by_key

import open_clip
clip_model, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=PRETRAINED, device=DEVICE)
clip_model.eval()
for p in clip_model.parameters(): p.requires_grad_(False)
extractor = OpenClipLocalExtractor(clip_model, LAYERS)


## Scoring Helpers

In [ ]:

def local_deviation_one(target_z, real_z, neighbor_radius=1):
    n = target_z.shape[0]
    side = int(round(math.sqrt(n)))
    if side * side != n:
        neighbor_radius = 0
    vals = []
    for i in range(n):
        if neighbor_radius > 0:
            r, c = divmod(i, side)
            js = []
            for rr in range(max(0, r - neighbor_radius), min(side, r + neighbor_radius + 1)):
                for cc in range(max(0, c - neighbor_radius), min(side, c + neighbor_radius + 1)):
                    js.append(rr * side + cc)
            candidates = real_z[:, js, :].reshape(-1, real_z.shape[-1])
        else:
            candidates = real_z[:, i, :]
        vals.append(1.0 - torch.matmul(candidates, target_z[i]).max())
    return torch.stack(vals)

def clone_memory(memory):
    return {
        'global': memory['global'].clone(),
        'local': {key: value.clone() for key, value in memory['local'].items()},
        'labels': memory['labels'].clone(),
        'metadata': dict(memory.get('metadata', {})),
    }

def append_real_memory(memory, target_global, target_local, indices):
    if len(indices) == 0:
        return memory
    idx = torch.as_tensor(indices, dtype=torch.long)
    memory['global'] = torch.cat([memory['global'], target_global[idx]], dim=0).contiguous()
    memory['local'] = {
        key: torch.cat([memory['local'][key], target_local[key][idx]], dim=0).contiguous()
        for key in memory['local']
    }
    memory['labels'] = torch.cat([memory['labels'], torch.zeros(len(idx), dtype=torch.long)], dim=0)
    return memory

def score_batch(target_global, target_local, memory):
    sims = target_global @ memory['global'].T
    nn_idx = sims.topk(k=min(RETRIEVE_M, memory['global'].shape[0]), dim=1).indices
    scores = []
    for row in range(target_global.shape[0]):
        layer_scores = []
        for key in target_local.keys():
            patch_d = local_deviation_one(target_local[key][row], memory['local'][key][nn_idx[row]], NEIGHBOR_RADIUS)
            layer_scores.append(float(patch_d.topk(min(TOP_K_PATCHES, patch_d.numel())).values.mean()))
        scores.append(float(np.mean(layer_scores)))
    return scores

@torch.inference_mode()
def predict_linear_probs(model, feats):
    return torch.sigmoid(model(feats.to(DEVICE))).detach().cpu().numpy().reshape(-1)

def gmm_threshold_and_fake_prob(scores, fit_scores=None):
    scores = np.asarray(scores, dtype=np.float64)
    fit_scores = scores if fit_scores is None else np.asarray(fit_scores, dtype=np.float64)
    if len(fit_scores) < 4 or np.unique(np.round(fit_scores, 12)).size < 2:
        threshold = float(np.median(fit_scores))
        probs = (scores >= threshold).astype(np.float64)
        return threshold, probs
    gmm = GaussianMixture(n_components=2, random_state=SEED).fit(fit_scores.reshape(-1, 1))
    means = gmm.means_.ravel()
    order = np.argsort(means)
    high = int(order[1])
    grid = np.linspace(fit_scores.min(), fit_scores.max(), 4096)
    logprob = gmm._estimate_weighted_log_prob(grid.reshape(-1, 1))
    diff = logprob[:, order[0]] - logprob[:, order[1]]
    between = (grid >= means[order[0]]) & (grid <= means[order[1]])
    if between.any():
        ids = np.where(between)[0]
        threshold = float(grid[ids[np.argmin(np.abs(diff[ids]))]])
    else:
        threshold = float(means.mean())
    return threshold, gmm.predict_proba(scores.reshape(-1, 1))[:, high]


def threshold_sigmoid_fake_prob(scores, threshold, fit_scores=None):
    scores = np.asarray(scores, dtype=np.float64)
    fit_scores = scores if fit_scores is None else np.asarray(fit_scores, dtype=np.float64)
    q25, q75 = np.quantile(fit_scores, [0.25, 0.75])
    scale = max(float((q75 - q25) / 2.0), 1e-6)
    logits = np.clip((scores - threshold) / scale, -50, 50)
    return 1.0 / (1.0 + np.exp(-logits))

def choose_ral_threshold_and_fake_prob(scores, *, non_safe_scores=None):
    scores = np.asarray(scores, dtype=np.float64)
    non_safe_scores = scores if non_safe_scores is None else np.asarray(non_safe_scores, dtype=np.float64)
    if RAL_THRESHOLD_MODE == 'gmm_non_safe':
        return gmm_threshold_and_fake_prob(scores, fit_scores=non_safe_scores)
    if RAL_THRESHOLD_MODE == 'gmm_all':
        return gmm_threshold_and_fake_prob(scores)
    if RAL_THRESHOLD_MODE == 'quantile_prior':
        fake_prior = float(RAL_FAKE_PRIOR)
        if not 0.0 < fake_prior < 1.0:
            raise ValueError(f'RAL_FAKE_PRIOR must be between 0 and 1, got {RAL_FAKE_PRIOR}')
        threshold = float(np.quantile(scores, 1.0 - fake_prior))
        return threshold, threshold_sigmoid_fake_prob(scores, threshold, fit_scores=scores)
    raise ValueError(f'Unknown RAL_THRESHOLD_MODE: {RAL_THRESHOLD_MODE}')

def calculate_eer(y_true, y_score):
    if np.unique(y_true).size < 2:
        return np.nan, np.nan
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[idx] + fnr[idx]) / 2), float(thresholds[idx])

def summarize(dataset, method, y_true, y_score, y_pred, extra=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    eer, eer_threshold = calculate_eer(y_true, y_score)
    row = {
        'dataset': dataset,
        'method': method,
        'acc': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'auc': roc_auc_score(y_true, y_score) if np.unique(y_true).size == 2 else np.nan,
        'ap': average_precision_score(y_true, y_score) if np.unique(y_true).size == 2 else np.nan,
        'eer': eer,
        'eer_threshold': eer_threshold,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }
    if extra:
        row.update(extra)
    return row


## Run CelebDF-v1 Level 4 Ensemble Logging


In [ ]:
all_sample_rows = []
summary_rows = []

try:
    for spec in TARGET_SPECS:
        print('\n===', spec['name'], '===')
        target_df = build_target_df(spec)
        print('rows:', len(target_df), '| counts:', target_df['label_num'].value_counts().sort_index().to_dict())
        if target_df.empty:
            print('skip empty target')
            continue

        loader = DataLoader(
            TargetImageDataset(target_df, preprocess),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
        )
        rows = []
        ral_scores_pass1 = []
        for images, labels, paths in tqdm(loader):
            images = images.to(DEVICE, non_blocking=True)
            target_global, target_local = extractor(images)
            ral_scores = score_batch(target_global, target_local, source_memory)
            ral_scores_pass1.extend(ral_scores)
            linear_probs = predict_linear_probs(linear_model, target_global)
            for path, label, linear_prob in zip(paths, labels.tolist(), linear_probs):
                rows.append({
                    'dataset': spec['name'],
                    'path': path,
                    'label': int(label),
                    'linear_prob': float(linear_prob),
                    'level': spec.get('level'),
                    'corruption': spec.get('corruption'),
                    'replacement_root': None if spec.get('replacement_root') is None else str(spec.get('replacement_root')),
                })

        df_scores = pd.DataFrame(rows)
        if df_scores.empty:
            print('skip empty score table')
            continue

        memory = clone_memory(source_memory)
        ral_scores_pass1 = np.asarray(ral_scores_pass1, dtype=np.float64)
        ral_scores = ral_scores_pass1
        target_real_added = 0
        safe_mask = np.zeros(len(df_scores), dtype=bool)
        if RUN_TARGET_EXPANSION and len(target_df) > 0:
            hflip_loader = DataLoader(
                TargetImageDataset(target_df, preprocess, augment='hflip'),
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=torch.cuda.is_available(),
            )
            hflip_scores = []
            for images, labels, paths in tqdm(hflip_loader, desc=f"{spec['name']} hflip for safe expansion"):
                images = images.to(DEVICE, non_blocking=True)
                hflip_global, hflip_local = extractor(images)
                hflip_scores.extend(score_batch(hflip_global, hflip_local, source_memory))
            hflip_scores = np.asarray(hflip_scores, dtype=np.float64)
            mean_scores = (ral_scores_pass1 + hflip_scores) / 2.0
            var_scores = np.var(np.stack([ral_scores_pass1, hflip_scores], axis=1), axis=1)
            safe_tau = float(np.quantile(mean_scores, EXPAND_SCORE_QUANTILE))
            safe = np.where((mean_scores < safe_tau) & (var_scores < EXPAND_MAX_AUG_VAR))[0]
            safe = safe[:MAX_TARGET_REAL_ADD]
            target_real_added = int(len(safe))
            safe_mask[safe] = True
            if target_real_added:
                safe_df = target_df.iloc[safe].reset_index(drop=True)
                safe_loader = DataLoader(
                    TargetImageDataset(safe_df, preprocess),
                    batch_size=BATCH_SIZE,
                    shuffle=False,
                    num_workers=NUM_WORKERS,
                    pin_memory=torch.cuda.is_available(),
                )
                for images, labels, paths in tqdm(safe_loader, desc=f"{spec['name']} append safe real"):
                    images = images.to(DEVICE, non_blocking=True)
                    safe_global, safe_local = extractor(images)
                    memory = append_real_memory(memory, safe_global, safe_local, range(safe_global.shape[0]))
            print('safe target real added:', target_real_added, '| tau_real:', safe_tau)
            ral_scores = []
            for images, labels, paths in tqdm(loader, desc=f"{spec['name']} RAL score pass2"):
                images = images.to(DEVICE, non_blocking=True)
                target_global, target_local = extractor(images)
                ral_scores.extend(score_batch(target_global, target_local, memory))
            ral_scores = np.asarray(ral_scores, dtype=np.float64)

        df_scores['ral_score_pass1'] = ral_scores_pass1
        df_scores['ral_score'] = ral_scores
        df_scores['ral_safe_memory_added'] = safe_mask.astype(int)

        threshold_fit_scores = df_scores.loc[~safe_mask, 'ral_score'].values if safe_mask.any() else df_scores['ral_score'].values
        ral_threshold, ral_prob = choose_ral_threshold_and_fake_prob(df_scores['ral_score'].values, non_safe_scores=threshold_fit_scores)
        df_scores['ral_prob'] = ral_prob
        df_scores['ral_pred'] = (df_scores['ral_score'] > ral_threshold).astype(int)
        df_scores['linear_pred'] = (df_scores['linear_prob'] >= LINEAR_PRED_THRESHOLD).astype(int)

        y = df_scores['label'].values
        common = {
            'level': spec.get('level'),
            'corruption': spec.get('corruption'),
            'replacement_root': None if spec.get('replacement_root') is None else str(spec.get('replacement_root')),
            'ral_threshold': ral_threshold,
            'linear_threshold': LINEAR_PRED_THRESHOLD,
            'target_real_added': target_real_added,
            'ral_threshold_mode': RAL_THRESHOLD_MODE,
            'ral_fake_prior': RAL_FAKE_PRIOR,
        }

        summary_rows.append(summarize(
            spec['name'],
            'ral_clip',
            y,
            df_scores['ral_prob'].values,
            df_scores['ral_pred'].values,
            {**common, 'probability_column': 'ral_prob', 'alpha': np.nan, 'threshold': ral_threshold},
        ))
        summary_rows.append(summarize(
            spec['name'],
            linear_model_type,
            y,
            df_scores['linear_prob'].values,
            df_scores['linear_pred'].values,
            {**common, 'probability_column': 'linear_prob', 'alpha': np.nan, 'threshold': LINEAR_PRED_THRESHOLD},
        ))

        for alpha in ENSEMBLE_ALPHAS:
            alpha_id = str(alpha).replace('.', 'p')
            prob_col = f'ensemble_prob_alpha_{alpha_id}'
            pred_col = f'ensemble_pred_alpha_{alpha_id}'
            df_scores[prob_col] = alpha * df_scores['ral_prob'].values + (1.0 - alpha) * df_scores['linear_prob'].values
            df_scores[pred_col] = (df_scores[prob_col] >= 0.5).astype(int)
            summary_rows.append(summarize(
                spec['name'],
                f'ensemble_alpha_{alpha_id}',
                y,
                df_scores[prob_col].values,
                df_scores[pred_col].values,
                {**common, 'probability_column': prob_col, 'alpha': alpha, 'threshold': 0.5},
            ))

        all_sample_rows.append(df_scores)
finally:
    extractor.close()

if not all_sample_rows:
    raise RuntimeError('No target scores were produced. Check Kaggle input paths above.')

samples = pd.concat(all_sample_rows, ignore_index=True)
summary = pd.DataFrame(summary_rows)

front_cols = [
    'dataset', 'path', 'label', 'level', 'corruption', 'replacement_root',
    'ral_score_pass1', 'ral_score', 'ral_prob', 'ral_safe_memory_added', 'linear_prob',
]
ensemble_prob_cols = [f'ensemble_prob_alpha_{str(alpha).replace(".", "p")}' for alpha in ENSEMBLE_ALPHAS]
pred_cols = ['ral_pred', 'linear_pred'] + [f'ensemble_pred_alpha_{str(alpha).replace(".", "p")}' for alpha in ENSEMBLE_ALPHAS]
samples = samples[front_cols + ensemble_prob_cols + pred_cols]

samples.to_csv(SAMPLES_OUTPUT, index=False)
summary.to_csv(SUMMARY_OUTPUT, index=False)
print('saved samples:', SAMPLES_OUTPUT)
print('saved summary:', SUMMARY_OUTPUT)

display(summary.sort_values(['dataset', 'method']))
display(samples.head())
